# How to use
- เปิด Playground mode ด้วยการกด `File` -> `Open in playground mode`
- พิมพ์เลขปีงบประมาณของงบที่ต้องการสร้าง csv ในช่อง `เลขปีงบประมาณ`
- copy url ของ Google Sheet ข้อมูลงบที่ validate แล้วมาใส่ในช่อง `url ของ Google Sheet Tree งบประมาณที่ validate แล้ว`
- copy url ของ Google Sheet ที่ต้องการสร้างตารางผลลัพธ์มาใส่ในช่อง `url ของ Google Sheet ที่ต้องการบันทึกผลลัพธ์`
- กด Run all
---
Note
- ในการรันครั้งแรกระบบจะขอให้ login ด้วย Google Account
- Login ด้วย account ที่มี permission แก้ไข [sheet สำหรับ validate](https://docs.google.com/spreadsheets/d/1zArSJZE1l3nuOMbPvnxNGmATLy1g1e0IjBcJkxJLFxU/edit?gid=51205943#gid=51205943?target=_blank)

In [ ]:
#@title เลขปีงบประมาณ
BUDGET_YEAR = None #@param {type:"integer"}

In [ ]:
#@title url ของ Google Sheet Tree งบประมาณที่ validate แล้ว
TREE_SHEET_URL = '' # @param {"type": "string"}

In [ ]:
#@title url ของ Google Sheet ที่ต้องการบันทึกผลลัพธ์
OUTPUT_SHEET_URL = '' # @param {"type": "string"}

# Code

In [4]:
import os
import re
import requests
import json
import pandas as pd
import datetime

## Authenticate to Google

In [5]:
# Connect to Google Account & Google Sheet
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

## Connect to Google Sheet

In [6]:
gc = gspread.authorize(creds)

In [7]:
# Get validation tree sheet
validate_sheet = gc.open_by_url(TREE_SHEET_URL)
validate_sheet

<Spreadsheet '[OpenBudget] test build final csv' id:1Ws1LvxM5nkCiibBO9wk5U-bQMsszA4xErScThj4_zSk>

In [8]:
# Get output sheet
output_sheet = gc.open_by_url(OUTPUT_SHEET_URL)
output_sheet

<Spreadsheet '[OpenBudget] test build final csv' id:1Ws1LvxM5nkCiibBO9wk5U-bQMsszA4xErScThj4_zSk>

## Clone & Install Validation Tool

In [9]:
!rm -rf wevis-openbudget-validation-tools && git clone https://github.com/wevisdemo/wevis-openbudget-validation-tools.git

Cloning into 'wevis-openbudget-validation-tools'...
remote: Enumerating objects: 367, done.
remote: Counting objects: 100% (367/367), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 367 (delta 173), reused 337 (delta 143), pack-reused 0 (from 0)
Receiving objects: 100% (367/367), 65.31 KiB | 2.11 MiB/s, done.
Resolving deltas: 100% (173/173), done.


In [10]:
!pip install -e "wevis-openbudget-validation-tools/open-budget-tree-validator" -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.8 MB/s eta 0:00:00
  Building editable for open_budget_tree_validator (pyproject.toml) ... done


In [11]:
import site

for sitepackages in site.getsitepackages():
    site.addsitedir(sitepackages)

# Build Final CSV

## Load ministry names

In [12]:
ministry_name_df = pd.read_csv("https://raw.githubusercontent.com/wevisdemo/wevis-openbudget-ocr/refs/heads/master/data/min_agc_names.csv")
ministry_name_df.head(5)

,min_name,agc_name
0,สำนักนายกรัฐมนตรี,สำนักงานปลัดสำนักนายกรัฐมนตรี
1,สำนักนายกรัฐมนตรี,กรมประชาสัมพันธ์
2,สำนักนายกรัฐมนตรี,สำนักงานคณะกรรมการคุ้มครองผู้บริโภค
3,สำนักนายกรัฐมนตรี,สำนักเลขาธิการนายกรัฐมนตรี
4,สำนักนายกรัฐมนตรี,สำนักเลขาธิการคณะรัฐมนตรี


In [13]:
ministries_name = ministry_name_df['min_name'].unique()
ministries_name

array(['สำนักนายกรัฐมนตรี', 'กระทรวงกลาโหม', 'กระทรวงการคลัง',
       'กระทรวงการต่างประเทศ', 'กระทรวงการท่องเที่ยวและกีฬา',
       'กระทรวงการพัฒนาสังคมและความมั่นคงของมนุษย์',
       'กระทรวงเกษตรและสหกรณ์', 'กระทรวงคมนาคม',
       'กระทรวงทรัพยากรธรรมชาติและสิ่งแวดล้อม',
       'กระทรวงดิจิทัลเพื่อเศรษฐกิจและสังคม', 'กระทรวงพลังงาน',
       'กระทรวงพาณิชย์', 'กระทรวงมหาดไทย', 'กระทรวงยุติธรรม',
       'กระทรวงแรงงาน', 'กระทรวงวัฒนธรรม', 'กระทรวงศึกษาธิการ',
       'กระทรวงสาธารณสุข', 'กระทรวงอุตสาหกรรม',
       'กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม',
       'ส่วนราชการไม่สังกัดสำนักนายกรัฐมนตรี กระทรวง หรือทบวง และหน่วยงานภายใต้การควบคุมดูแลของนายกรัฐมนตรี',
       'หน่วยงานของรัฐสภา', 'หน่วยงานของศาล',
       'หน่วยงานขององค์กรอิสระและองค์กรอัยการ', 'รัฐวิสาหกิจ',
       'ส่วนราชการในพระองค์', 'สภากาชาดไทย', 'หน่วยงานอื่นของรัฐ',
       'จังหวัดและกลุ่มจังหวัด', 'องค์กรปกครองส่วนท้องถิ่น',
       'ทุนหมุนเวียน', 'กองทุนและเงินทุนหมุนเวียน', 'งบกลาง',
       'รายจ่ายเพื่

# Build DF for each Ministry

In [14]:
from open_budget_tree_validator import BudgetTree

In [15]:
added_ministries = set()

final_df = None
for sheet in validate_sheet.worksheets():
  sheet_title = sheet.title

  matched_ministry_name = re.search(
      r"(" + "|".join(ministries_name) + ")",
      sheet_title
  )
  if matched_ministry_name:
    if matched_ministry_name.group(1) not in added_ministries:
      sheet_values = sheet.get_all_values()
      df = pd.DataFrame(sheet_values[1:], columns=sheet_values[0])
      budget_tree = BudgetTree(df)
      added_ministries.add(matched_ministry_name.group(1))

      if final_df is None:
        final_df = budget_tree.build_final_df(BUDGET_YEAR)
        continue

      final_df = pd.concat([
          final_df,
          budget_tree.build_final_df(BUDGET_YEAR)
      ])

/content/wevis-openbudget-validation-tools/open-budget-tree-validator/src/open_budget_tree_validator/final_df_builder.py:162: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  process_df = pd.concat(
/content/wevis-openbudget-validation-tools/open-budget-tree-validator/src/open_budget_tree_validator/final_df_builder.py:162: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  process_df = pd.concat(
/content/wevis-openbudget-validation-tools/open-budget-tree-validator/src/open_budget_tree_validator/budget_

In [16]:
final_df.head(5)

,REF_DOC,REF_PAGE_NO,MINISTRY,BUDGETARY_UNIT,BUDGET_PLAN,CROSS_FUNC?,OUTPUT,PROJECT,CATEGORY_LV1,CATEGORY_LV2,CATEGORY_LV3,CATEGORY_LV4,CATEGORY_LV5,CATEGORY_LV6,CATEGORY_LV7,ITEM_DESCRIPTION,AMOUNT,FISCAL_YEAR,OBLIGED?
0,./ฉบับร่างพระราชบัญญัติงบประมาณรายจ่าย (ร่าง พ...,6,สภากาชาดไทย,สภากาชาดไทย,แผนงานยุทธศาสตร์เสริมสร้างให้คนมีสุขภาวะที่ดี,False,การบริการรักษาพยาบาล ฟื้นฟูสภาพ สร้างเสริมสุขภ...,,เงินอุดหนุน,เงินอุดหนุนทั่วไป,,,,,,ค่าใช้จ่ายบุคลากร,5.318618e+09,2027.0,False
1,./ฉบับร่างพระราชบัญญัติงบประมาณรายจ่าย (ร่าง พ...,6,สภากาชาดไทย,สภากาชาดไทย,แผนงานยุทธศาสตร์เสริมสร้างให้คนมีสุขภาวะที่ดี,False,การบริการรักษาพยาบาล ฟื้นฟูสภาพ สร้างเสริมสุขภ...,,เงินอุดหนุน,เงินอุดหนุนทั่วไป,,,,,,ค่าใช้จ่ายดำเนินงาน,9.969199e+08,2027.0,False
2,./ฉบับร่างพระราชบัญญัติงบประมาณรายจ่าย (ร่าง พ...,6,สภากาชาดไทย,สภากาชาดไทย,แผนงานยุทธศาสตร์เสริมสร้างให้คนมีสุขภาวะที่ดี,False,การบริการรักษาพยาบาล ฟื้นฟูสภาพ สร้างเสริมสุขภ...,,เงินอุดหนุน,เงินอุดหนุนทั่วไป,ค่าครุภัณฑ์,,,,,ครุภัณฑ์คอมพิวเตอร์ที่มีราคาต่อหน่วยต่ำกว่า 1 ...,9.630000e+05,2027.0,False
3,./ฉบับร่างพระราชบัญญัติงบประมาณรายจ่าย (ร่าง พ...,6,สภากาชาดไทย,สภากาชาดไทย,แผนงานยุทธศาสตร์เสริมสร้างให้คนมีสุขภาวะที่ดี,False,การบริการรักษาพยาบาล ฟื้นฟูสภาพ สร้างเสริมสุขภ...,,เงินอุดหนุน,เงินอุดหนุนทั่วไป,ค่าครุภัณฑ์,,,,,ครุภัณฑ์การแพทย์ที่มีราคาต่อหน่วยต่ำกว่า 1 ล้า...,9.899580e+07,2027.0,False
4,./ฉบับร่างพระราชบัญญัติงบประมาณรายจ่าย (ร่าง พ...,6,สภากาชาดไทย,สภากาชาดไทย,แผนงานยุทธศาสตร์เสริมสร้างให้คนมีสุขภาวะที่ดี,False,การบริการรักษาพยาบาล ฟื้นฟูสภาพ สร้างเสริมสุขภ...,,เงินอุดหนุน,เงินอุดหนุนทั่วไป,ค่าครุภัณฑ์,,,,,เครื่องช่วยหายใจในหอผู้ป่วยวิกฤติ แขวงปทุมวัน ...,5.000000e+07,2027.0,False


# Add new Sheet

In [17]:
import zoneinfo

bangkok_tz = zoneinfo.ZoneInfo("Asia/Bangkok")
now_bangkok = datetime.datetime.now(bangkok_tz)
sheet_title = now_bangkok.date().isoformat() + '#'

# Check if sheet exist
if sheet_title in [_.title for _ in output_sheet.worksheets()]:
  sheet_title += now_bangkok.strftime("%H%M%S")
new_sheet = output_sheet.add_worksheet(
    title=sheet_title,
    rows=len(final_df.index),
    cols=len(final_df.columns)
)
new_sheet.update(
    [final_df.columns.values.tolist()] + final_df.values.tolist(),
    value_input_option='USER_ENTERED'
)

{'spreadsheetId': '1Ws1LvxM5nkCiibBO9wk5U-bQMsszA4xErScThj4_zSk',
 'updatedRange': "'2026-10-06#113436'!A1:S866",
 'updatedRows': 866,
 'updatedColumns': 19,
 'updatedCells': 16454}